# E3-K — Évaluation rolling-origin de la généralisation temporelle (15 min)

**But :** compléter l'audit descriptif `09_temporal_generalization_shift` par une évaluation **avec réentraînement sur plusieurs origines temporelles**, sans écraser les résultats existants.

**Portée exacte :** baseline Persistence, Ridge et Random Forest sur les **mêmes 19 variables tabulaires locales**, à horizon **15 minutes uniquement**. Ce contrôle ciblé ne prétend pas valider tous les modèles ni les quatre horizons de E3. Les folds réutilisent les données historiques (et peuvent recouvrir les périodes de test antérieures) : leurs résultats constituent une **analyse rétrospective de sensibilité temporelle**, distincte du test final E3 gelé.

**Précautions :** partitions chronologiques sur `target_timestamp`, séparation des fenêtres par un embargo de 15 minutes, préparation ajustée uniquement sur le train, validation distincte, aucun choix de modèle sur le test, résultats par fold et par dispositif. Pas d'affirmation de causalité du décalage.


In [ ]:
from pathlib import Path
import json, platform, time
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline

ROOT=Path.cwd()
INPUT=ROOT/'outputs'/'07A_tabular'/'01_tabular_all_unscaled.csv'
if not INPUT.exists():
    candidates=list((ROOT/'outputs').rglob('01_tabular_all_unscaled.csv'))
    if len(candidates)!=1:
        raise FileNotFoundError(f'Fichier absent ou ambigu : {INPUT}; candidats={candidates}')
    INPUT=candidates[0]
OUT=ROOT/'outputs'/'E3_K_ROLLING_ORIGIN_TEMPORAL'
OUT.mkdir(parents=True,exist_ok=True)
SEED=42
HORIZON_MIN=15
VAL_DAYS=10
TEST_DAYS=14
N_FOLDS=4
MIN_TRAIN_DAYS=45
EMBARGO=pd.Timedelta(minutes=15)
print('Input:',INPUT)


In [ ]:
df=pd.read_csv(INPUT)
required={'uuid','timestamp','target_timestamp','segment_id','temperature','target_temperature_t_plus_15'}
assert required.issubset(df.columns),required-set(df.columns)
df['timestamp']=pd.to_datetime(df['timestamp'],utc=True)
df['target_timestamp']=pd.to_datetime(df['target_timestamp'],utc=True)
assert df['timestamp'].notna().all() and df['target_timestamp'].notna().all()
assert (df['target_timestamp']-df['timestamp']==pd.Timedelta(minutes=15)).all()
assert not df.duplicated(['uuid','timestamp']).any()
assert df['target_temperature_t_plus_15'].notna().all()
feature_cols=[c for c in df.columns if c not in
              {'uuid','timestamp','target_timestamp','segment_id','split','target_temperature_t_plus_15'}]
assert len(feature_cols)==19,('Unexpected number of features',feature_cols)
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in feature_cols)
assert df['uuid'].nunique()==7,df['uuid'].nunique()
df=df.sort_values(['timestamp','uuid']).reset_index(drop=True)
print('Rows',len(df),'range',df.timestamp.min(),df.target_timestamp.max())
print('Features:',feature_cols)


## Construction des folds

Quatre fenêtres de test successives de 14 jours sont placées en fin de période ; pour chaque fold, la validation couvre les 10 jours précédents et le train est **expansif**. Les lignes sont sélectionnées en fonction de l'heure de la **cible** : ainsi, aucune cible d'entraînement ne chevauche une période de validation ou de test. Les caractéristiques sont disponibles à l'origine de prévision, avant la cible.


In [ ]:
# Align boundaries to midnight UTC to make fold dates reproducible.
end_exclusive=df.target_timestamp.max().floor('D')+pd.Timedelta(days=1)
folds=[]
for k in range(N_FOLDS):
    test_end=end_exclusive-pd.Timedelta(days=(N_FOLDS-1-k)*TEST_DAYS)
    test_start=test_end-pd.Timedelta(days=TEST_DAYS)
    val_end=test_start-EMBARGO
    val_start=val_end-pd.Timedelta(days=VAL_DAYS)
    train_end=val_start-EMBARGO
    train_start=df.target_timestamp.min()
    assert train_end-train_start>=pd.Timedelta(days=MIN_TRAIN_DAYS), 'Historique insuffisant'
    # Half-open intervals, with explicit embargo between target ranges.
    masks={
      'train':(df.target_timestamp>=train_start)&(df.target_timestamp<train_end),
      'validation':(df.target_timestamp>=val_start)&(df.target_timestamp<val_end),
      'test':(df.target_timestamp>=test_start)&(df.target_timestamp<test_end)
    }
    sets={s:df.loc[m].copy() for s,m in masks.items()}
    assert all(len(v)>100 for v in sets.values()),(k,{s:len(v) for s,v in sets.items()})
    assert max(sets['train'].target_timestamp)<min(sets['validation'].target_timestamp)
    assert max(sets['validation'].target_timestamp)<min(sets['test'].target_timestamp)
    assert all(v.uuid.nunique()==7 for v in sets.values())
    folds.append((k+1,sets))
audit=[]
for fold,sets in folds:
    for split,d in sets.items():
        audit.append({'fold':fold,'split':split,'n':len(d),'devices':d.uuid.nunique(),
                      'start_target_utc':d.target_timestamp.min().isoformat(),
                      'end_target_utc':d.target_timestamp.max().isoformat(),
                      'n_unique_timestamps':d.timestamp.nunique()})
audit_df=pd.DataFrame(audit)
audit_df.to_csv(OUT/'01_fold_time_coverage.csv',index=False)
print(audit_df.to_string(index=False))


## Modèles et métriques

Les hyperparamètres sont fixés avant l'évaluation des folds (`Ridge alpha=1`, `RF n_estimators=200`, `min_samples_leaf=2`, `max_features=1.0`, `random_state=42`). Ils ne sont pas optimisés sur les tests. La validation sert de diagnostic et non de sélection opportuniste. Toutes les configurations utilisent les mêmes lignes et cibles. Les prédictions Persistence utilisent `temperature(t)`.


In [ ]:
rows=[];preds=[];runtime=[]
for fold,sets in folds:
    train,val,test=(sets[x] for x in ('train','validation','test'))
    Xtr=train[feature_cols];ytr=train.target_temperature_t_plus_15.to_numpy()
    models={
      'Persistence':None,
      'Ridge':Pipeline([('imputer',SimpleImputer(strategy='median')),
                        ('scaler',StandardScaler()),('reg',Ridge(alpha=1.0))]),
      'RF':Pipeline([('imputer',SimpleImputer(strategy='median')),
                     ('reg',RandomForestRegressor(n_estimators=200,min_samples_leaf=2,
                                                   max_features=1.0,n_jobs=-1,random_state=SEED))])
    }
    for name,model in models.items():
        start=time.perf_counter()
        if model is not None:model.fit(Xtr,ytr)
        elapsed=time.perf_counter()-start
        runtime.append({'fold':fold,'model':name,'fit_seconds':elapsed})
        for split,d in (('validation',val),('test',test)):
            y=d.target_temperature_t_plus_15.to_numpy()
            pred=(d.temperature.to_numpy() if model is None else model.predict(d[feature_cols]))
            assert len(y)==len(pred) and np.isfinite(pred).all()
            err=y-pred
            rows.append({'fold':fold,'split':split,'model':name,'n':len(y),
                         'RMSE':float(np.sqrt(np.mean(err**2))),
                         'MAE':float(np.mean(np.abs(err))),
                         'Bias':float(np.mean(pred-y))})
            if split=='test':
                temp=d[['uuid','timestamp','target_timestamp','segment_id']].copy()
                temp['fold']=fold;temp['model']=name
                temp['y_true']=y;temp['y_pred']=pred;temp['abs_error']=np.abs(err)
                preds.append(temp)
    print('Fold',fold,'finished')
metrics=pd.DataFrame(rows)
predictions=pd.concat(preds,ignore_index=True)
assert len(metrics)==N_FOLDS*3*2
assert not predictions.duplicated(['fold','model','uuid','timestamp']).any()
metrics.to_csv(OUT/'02_metrics_by_fold_split.csv',index=False)
predictions.to_csv(OUT/'03_test_predictions_all_folds.csv',index=False)
pd.DataFrame(runtime).to_csv(OUT/'04_fit_runtime.csv',index=False)
print(metrics[metrics.split=='test'].pivot(index='fold',columns='model',values='RMSE').round(5))


In [ ]:
# Device-level metrics and fold-to-fold variability. Note: overlapping expanding
# train windows and successive test windows do not imply independent folds.
dev=[]
for (fold,model,uuid),d in predictions.groupby(['fold','model','uuid'],sort=True):
    dev.append({'fold':fold,'model':model,'uuid':uuid,'n':len(d),
                'RMSE':float(np.sqrt(np.mean((d.y_true-d.y_pred)**2))),
                'MAE':float(np.mean(d.abs_error))})
pd.DataFrame(dev).to_csv(OUT/'05_device_metrics_by_fold.csv',index=False)
test_metrics=metrics[metrics.split=='test']
summary=(test_metrics.groupby('model',as_index=False)
         .agg(folds=('fold','nunique'),mean_fold_RMSE=('RMSE','mean'),
              sd_fold_RMSE=('RMSE','std'),min_fold_RMSE=('RMSE','min'),
              max_fold_RMSE=('RMSE','max'),mean_fold_MAE=('MAE','mean')))
summary.to_csv(OUT/'00_PRIMARY_K_rolling_summary.csv',index=False)
wide=test_metrics.pivot(index='fold',columns='model',values='RMSE')
comparisons=pd.DataFrame({'fold':wide.index,
                          'Ridge_minus_Persistence':wide['Ridge'].to_numpy()-wide['Persistence'].to_numpy(),
                          'RF_minus_Persistence':wide['RF'].to_numpy()-wide['Persistence'].to_numpy(),
                          'RF_minus_Ridge':wide['RF'].to_numpy()-wide['Ridge'].to_numpy()})
comparisons.to_csv(OUT/'06_paired_fold_RMSE_differences.csv',index=False)
protocol={'experiment':'E3-K rolling-origin retrospective temporal evaluation',
          'input':str(INPUT),'horizon_minutes':HORIZON_MIN,'n_folds':N_FOLDS,
          'test_days':TEST_DAYS,'validation_days':VAL_DAYS,
          'min_train_days':MIN_TRAIN_DAYS,'embargo_minutes':15,
          'features':feature_cols,'models':['Persistence','Ridge(alpha=1)','RF(200 trees,min_samples_leaf=2)'],
          'seed':SEED,'data_overlap_note':'retrospective folds may include historical held-out data from earlier experiments; no external independent test',
          'limitations':['single 15-min horizon','only three baseline pipelines',
                         'four correlated expanding-training folds',
                         'hyperparameters fixed rather than retuned per fold',
                         'no confidence interval inferred from four folds'],
          'python':platform.python_version(),'pandas':pd.__version__}
(OUT/'07_protocol.json').write_text(json.dumps(protocol,indent=2,ensure_ascii=False),encoding='utf-8')
print(summary.to_string(index=False))
print('Outputs:',OUT)


## Interprétation pour le Reviewer 4

Comparer **les quatre RMSE de test par modèle**, la dispersion entre folds, et les différences appariées par fold. Ne pas présenter les quatre folds comme des tests indépendants ni extrapoler à d'autres bâtiments. Une diminution de performance au fil des folds peut refléter une évolution des régimes thermiques, mais ne démontre pas une causalité du shift. Le contrôle couvre uniquement la prévision à 15 min avec entrées tabulaires locales.

**Fichiers à transmettre :** dossier ZIP `outputs/E3_K_ROLLING_ORIGIN_TEMPORAL` contenant `00_PRIMARY_K_rolling_summary.csv`, `01_fold_time_coverage.csv`, `02_metrics_by_fold_split.csv`, `03_test_predictions_all_folds.csv`, `05_device_metrics_by_fold.csv`, `06_paired_fold_RMSE_differences.csv`, `07_protocol.json`.
